# BIAN 310 | Module 4 Assignment 4.2 · Data Visualization Dashboard

**Instructor demonstration and student exercise**. The original Assignment 4.2 remains unchanged. Build a manager-friendly **one-page Excel dashboard** with at least four business questions, three KPIs, three chart types, three insights, and a management recommendation.

- Check **DEMO_MODE** for the original nine people supplied in Assignment 4.1.
- Uncheck **DEMO_MODE** for a new 240-person fictional practice run.
- Run every step. The last cell downloads one ZIP containing the Excel dashboard, data, charts, Word worksheet, and an optional Looker Studio CSV.

This dataset has **no time/date field**, so a historical trend line would be misleading. We will use sales rankings, a sales-versus-bonus scatter plot, and a tenure-band sales-share donut chart instead. Learners must rewrite the automated **DRAFT** insights and recommendation after checking the data.


In [ ]:
# STEP 1 | Enter your details and select DEMO or STUDENT mode.
DEMO_MODE = True #@param {type:"boolean"}
STUDENT_NAME = "" #@param {type:"string"}
ASSIGNMENT = "Module 4: Assignment 4.2 — Data Visualization Dashboard" #@param {type:"string"}

if not STUDENT_NAME.strip(): STUDENT_NAME = input('Enter your first and last name: ').strip()
if not ASSIGNMENT.strip(): ASSIGNMENT = input('Enter assignment title: ').strip()
if not STUDENT_NAME: raise ValueError('A student name is required.')
print('Student:',STUDENT_NAME,'\nAssignment:',ASSIGNMENT,'\nMode:','DEMO' if DEMO_MODE else 'STUDENT PRACTICE')


In [ ]:
# STEP 2 | Install missing packages automatically.
# This resolves the ModuleNotFoundError: No module named 'xlsxwriter' issue.
import importlib.util, subprocess, sys
deps={'pandas':'pandas','numpy':'numpy','matplotlib':'matplotlib','xlsxwriter':'XlsxWriter','docx':'python-docx'}
missing=[pkg for mod,pkg in deps.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from docx import Document
from docx.shared import Inches,Pt
from IPython.display import display,Image
from zipfile import ZipFile,ZIP_DEFLATED
import random,re
try: IN_COLAB=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:IN_COLAB=False
print('Packages ready. Colab:',IN_COLAB)


In [ ]:
# STEP 3 | Load one of two built-in datasets. No upload is needed.
FIELDS=['Salesperson','Total Sales ($)','Avg. Previous Bonus ($)','Customer Accounts','Years with Company']
ORIGINAL=[
 ('Smith, Michael',325000.78,12499.3452,124,14),('Yu, Joe',13678.21,239.9434,9,7),
 ('Reeves, Bill',452359.19,21987.2462,175,21),('Hamilton, Joshua',87423.91,7642.9011,28,3),
 ('Harper, Derek',87654.21,1250.1393,21,4),('Quinn, Dorothy',234091.39,14567.9833,48,9),
 ('Graves, Lorrie',379401.94,27981.4432,121,12),('Sun, Yi',31733.59,672.9111,7,1),
 ('Thompson, Nicole',127845.22,13322.9713,17,3)]
FIRST=['Alex','Avery','Blair','Casey','Cameron','Drew','Emery','Finley','Harper','Jordan','Kai','Kendall','Logan','Morgan','Parker','Quinn','Reese','Riley','Rowan','Sage','Skyler','Taylor','Terry','Dakota']
LAST=['Anderson','Baker','Carter','Diaz','Ellis','Foster','Garcia','Hayes','Irwin','Jones','Kim','Lewis','Mitchell','Nelson','Owens','Patel','Reed','Santos','Turner','Vega','Walker','Young','Zimmerman','Brooks']
def generate_practice():
    rng=random.Random(31041); rows=[]
    for j in range(240):
        name=f'{LAST[j//10]}, {FIRST[j%24]} {j//24+1:02d}'
        yrs=rng.randint(1,24);accounts=rng.randint(8,150)
        sales=max(11000,accounts*rng.uniform(1300,2900)+yrs*rng.uniform(1000,4900)+rng.gauss(0,38000))
        if j%37==0:sales*=1.36
        bonus=max(0,sales*rng.uniform(.026,.074)+rng.gauss(0,4700))
        rows.append((name,round(sales,2),round(bonus,4),accounts,yrs))
    return rows
original=pd.DataFrame(ORIGINAL if DEMO_MODE else generate_practice(),columns=FIELDS)
print('Rows:',len(original));display(original.head())


In [ ]:
# STEP 4 | Audit, preserve raw data and prepare a clean, sorted copy.
clean=original.copy()
clean['Salesperson']=clean['Salesperson'].astype(str).str.strip()
for col in FIELDS[1:]:clean[col]=pd.to_numeric(clean[col],errors='coerce')
checks=pd.DataFrame([
 {'Test':'Original rows','Count':len(original)},
 {'Test':'Missing data cells','Count':int(clean.isna().sum().sum())},
 {'Test':'Duplicate salesperson names','Count':int(clean['Salesperson'].duplicated().sum())},
 {'Test':'Negative numeric cells','Count':int((clean[FIELDS[1:]]<0).sum().sum())}])
if checks.loc[checks['Test'].isin(['Missing data cells','Negative numeric cells']),'Count'].sum():
    raise ValueError('Invalid source values require a human review.')
clean=clean.sort_values('Total Sales ($)',ascending=False).reset_index(drop=True)
clean['Tenure Band']=pd.cut(clean['Years with Company'],bins=[-1,4,9,14,100],labels=['0–4 years','5–9 years','10–14 years','15+ years'])
display(checks)


In [ ]:
# STEP 5 | State FOUR business questions before showing any dashboard.
business_questions=[
 'Q1. Who are the highest and lowest sales performers?',
 'Q2. Is prior bonus associated with total sales?',
 'Q3. Which tenure groups account for the greatest sales share?',
 'Q4. Which employees or groups warrant management review, and what evidence is missing?']
for question in business_questions:print(question)
print('These will appear on the Excel dashboard and the Word guide.')


In [ ]:
# STEP 6 | Compute five KPIs and explicit supporting evidence.
kpis={
 'Total Sales':float(clean['Total Sales ($)'].sum()),
 'Average Sales':float(clean['Total Sales ($)'].mean()),
 'Highest Sales':float(clean['Total Sales ($)'].max()),
 'Customer Accounts':int(clean['Customer Accounts'].sum()),
 'Average Previous Bonus':float(clean['Avg. Previous Bonus ($)'].mean())}
bonus_r=clean['Total Sales ($)'].corr(clean['Avg. Previous Bonus ($)'])
groups=clean.groupby('Tenure Band',observed=False)['Total Sales ($)'].sum().reindex(['0–4 years','5–9 years','10–14 years','15+ years']).fillna(0)
group_count=clean.groupby('Tenure Band',observed=False).size().reindex(groups.index,fill_value=0)
leader=clean.iloc[0];bottom=clean.iloc[-1];leading_band=groups.idxmax()
for k,v in kpis.items():print(k,':',f'{v:,.0f}' if k=='Customer Accounts' else f'${v:,.2f}')
print('Sales / bonus correlation:',round(bonus_r,2))
display(pd.DataFrame({'Total Sales':groups,'People':group_count}))


In [ ]:
# STEP 7 | Three DIFFERENT chart formats: bar, scatter and donut.
out=Path('/content/BIAN310_Assignment42_Outputs') if IN_COLAB else Path.cwd()/'BIAN310_Assignment42_Outputs'
out.mkdir(parents=True,exist_ok=True)
NAVY='#14344d';BLUE='#1b76aa';TEAL='#0c9d93';GOLD='#ebad48'
plt.rcParams.update({'figure.dpi':140,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def export_chart(name,fig):fig.savefig(out/name,bbox_inches='tight',facecolor='white');plt.close(fig)

fig,ax=plt.subplots(figsize=(10,6))
ranked=clean.head(12).iloc[::-1]
ax.barh(ranked['Salesperson'],ranked['Total Sales ($)']/1000,color=BLUE)
ax.set(xlabel='Total sales ($000s)',title='Chart 1 · Sales Ranking'+(' (top 12)' if len(clean)>12 else ''))
ax.grid(axis='x',alpha=.15);export_chart('01_sales_ranking.png',fig)

fig,ax=plt.subplots(figsize=(9,5.5))
ax.scatter(clean['Total Sales ($)']/1000,clean['Avg. Previous Bonus ($)']/1000,color=TEAL,alpha=.68,s=52,edgecolors='white',linewidths=.5)
ax.set(xlabel='Sales ($000s)',ylabel='Previous bonus ($000s)',title='Chart 2 · Sales and Previous Bonus')
ax.grid(alpha=.15);export_chart('02_sales_bonus_scatter.png',fig)

fig,ax=plt.subplots(figsize=(8,5.5))
active=groups[groups>0]
ax.pie(active.values,labels=active.index,autopct='%1.1f%%',startangle=90,pctdistance=.79,
       wedgeprops={'width':.38,'edgecolor':'white'},colors=[BLUE,TEAL,GOLD,'#6c7ab4'][:len(active)])
ax.set_title('Chart 3 · Sales Share by Tenure Band')
export_chart('03_sales_share_donut.png',fig)
print('Three chart types generated.')


In [ ]:
# STEP 8 | Generate DRAFT insights students must validate and rewrite.
insights=[
 f'DRAFT Insight 1: {leader["Salesperson"]} has the highest sales (${leader["Total Sales ($)"]:,.0f}), while {bottom["Salesperson"]} has the lowest (${bottom["Total Sales ($)"]:,.0f}). Compare quotas, territories and account potential.',
 f'DRAFT Insight 2: Sales and prior bonus show a linear correlation of r={bonus_r:.2f} in this sample. This does not explain historic bonus policy or justify a future award.',
 f'DRAFT Insight 3: Employees with {leading_band} tenure account for the largest share of combined sales ({groups.max()/groups.sum():.1%}); there are {group_count.loc[leading_band]} people in that group, so cohort size matters.']
recommendation=('DRAFT Recommendation: Management should review the strongest and weakest sales outcomes alongside targets, territory potential, '
 'profit margin, account retention, compensation policy and performance period. Do not set bonuses from this dashboard alone.')
for x in insights:print(x+'\n')
print(recommendation)


In [ ]:
# STEP 9 | Build and display the visual preview (fits a 16:9 screen).
fig=plt.figure(figsize=(16,9),facecolor='#f7fafc');g=GridSpec(3,6,figure=fig,height_ratios=[1,3.1,1.6],hspace=.4,wspace=.52)
fig.text(.045,.962,'MODULE 4.2  |  SALES MANAGER DASHBOARD',fontsize=20,color=NAVY,fontweight='bold')
fig.text(.045,.929,('ORIGINAL DEMO' if DEMO_MODE else 'FICTIONAL PRACTICE')+'  •  '+STUDENT_NAME,fontsize=10,color='#587085')
for j,(name,value) in enumerate(kpis.items()):
 x=.047+j*.184;fig.text(x,.86,name,fontsize=10,color='#526a7e')
 fig.text(x,.825,f'{value:,.0f}' if name=='Customer Accounts' else f'${value:,.0f}',fontsize=16,color=NAVY,fontweight='bold')
a=fig.add_subplot(g[1,0:2]);top=clean.head(9).iloc[::-1];a.barh(top['Salesperson'],top['Total Sales ($)']/1000,color=BLUE);a.set_title('Sales rank',loc='left');a.set_xlabel('Sales ($000s)')
b=fig.add_subplot(g[1,2:4]);b.scatter(clean['Total Sales ($)']/1000,clean['Avg. Previous Bonus ($)']/1000,color=TEAL,alpha=.65);b.set_title('Sales vs bonus',loc='left');b.set_xlabel('Sales ($000s)');b.set_ylabel('Bonus ($000s)')
c=fig.add_subplot(g[1,4:6]);active=groups[groups>0];c.pie(active.values,labels=active.index,autopct='%1.0f%%',startangle=90,wedgeprops={'width':.43},textprops={'fontsize':8});c.set_title('Sales by tenure',loc='left')
ins_ax=fig.add_subplot(g[2,:]);ins_ax.axis('off')
short=[f'1. Highest: {leader["Salesperson"]}; lowest: {bottom["Salesperson"]}.',
       f'2. Prior bonus association is r={bonus_r:.2f}; relationship alone cannot decide bonuses.',
       f'3. Largest tenure group share: {leading_band}, {groups.max()/groups.sum():.1%} of revenue.',
       'Recommendation: review profitability, targets, territory and retention before any bonus decision.']
for k,line in enumerate(short):ins_ax.text(.01,.84-k*.24,line,fontsize=12,color=NAVY)
fig.savefig(out/'Manager_Dashboard_Preview.png',bbox_inches='tight',facecolor=fig.get_facecolor());plt.close(fig)
display(Image(filename=str(out/'Manager_Dashboard_Preview.png'),width=1000))


In [ ]:
# STEP 10 | Build one printable Excel dashboard, plus source and analysis tabs.
excel=out/'Assignment42_One_Page_Excel_Dashboard.xlsx'
with pd.ExcelWriter(excel,engine='xlsxwriter') as writer:
    clean.to_excel(writer,sheet_name='Cleaned Data',index=False,startrow=2)
    original.to_excel(writer,sheet_name='Original Data',index=False,startrow=2)
    checks.to_excel(writer,sheet_name='Audit',index=False,startrow=2)
    book=writer.book;head=book.add_format({'bold':True,'bg_color':NAVY,'font_color':'#ffffff','text_wrap':True})
    title=book.add_format({'bold':True,'font_size':17,'font_color':NAVY})
    muted=book.add_format({'font_color':'#4d6576','text_wrap':True,'valign':'top'})
    text=book.add_format({'font_color':NAVY,'text_wrap':True,'valign':'top'})
    money=book.add_format({'num_format':'$#,##0.00'});number=book.add_format({'num_format':'#,##0'})
    large=book.add_format({'bold':True,'font_size':16,'font_color':'#0b8d7d','num_format':'$#,##0'})
    largeint=book.add_format({'bold':True,'font_size':16,'font_color':'#0b8d7d','num_format':'#,##0'})
    for name,frame in [('Cleaned Data',clean),('Original Data',original),('Audit',checks)]:
        sheet=writer.sheets[name];sheet.write(0,0,name,title);sheet.freeze_panes(3,1)
        sheet.set_column('A:A',30);sheet.set_column('B:C',23,money);sheet.set_column('D:E',21,number)
        for j,col in enumerate(frame.columns):sheet.write(2,j,col,head)
        if len(frame):sheet.autofilter(2,0,len(frame)+2,len(frame.columns)-1)
    dash=book.add_worksheet('Manager Dashboard');dash.activate();dash.set_first_sheet()
    dash.hide_gridlines(2);dash.set_landscape();dash.fit_to_pages(1,1)
    dash.set_paper(9);dash.set_margins(.25,.25,.35,.35)
    dash.print_area('A1:U58');dash.set_zoom(80)
    dash.set_column('A:A',2.8);dash.set_column('B:U',11.5)
    dash.set_default_row(17)
    dash.merge_range('B2:U3','ASSIGNMENT 4.2  |  SALES PERFORMANCE DASHBOARD',title)
    dash.merge_range('B4:U4',('ORIGINAL 9-PERSON DEMO' if DEMO_MODE else '240-PERSON STUDENT PRACTICE')+'  •  '+STUDENT_NAME,muted)
    refs=['B6','F6','J6','N6','R6'];end=len(clean)+3
    for j,(name,value) in enumerate(kpis.items()):
        col=[1,5,9,13,17][j];dash.merge_range(5,col,5,col+2,name,head)
        formula=[f"SUM('Cleaned Data'!B4:B{end})",f"AVERAGE('Cleaned Data'!B4:B{end})",
                 f"MAX('Cleaned Data'!B4:B{end})",f"SUM('Cleaned Data'!D4:D{end})",
                 f"AVERAGE('Cleaned Data'!C4:C{end})"][j]
        dash.write_formula(6,col,'='+formula,largeint if j==3 else large,value)
    dash.merge_range('B9:U9','FOUR BUSINESS QUESTIONS',head)
    for i,q in enumerate(business_questions):dash.merge_range(9+i,1,9+i,20,q,text)
    # Images reside within A14:U43 on a single landscape sheet.
    dash.insert_image('B15',str(out/'01_sales_ranking.png'),{'x_scale':.38,'y_scale':.40})
    dash.insert_image('I15',str(out/'02_sales_bonus_scatter.png'),{'x_scale':.38,'y_scale':.42})
    dash.insert_image('P15',str(out/'03_sales_share_donut.png'),{'x_scale':.36,'y_scale':.41})
    dash.merge_range('B39:U39','THREE DRAFT INSIGHTS  |  STUDENT: VALIDATE AND REWRITE',head)
    for i,ins in enumerate(insights):dash.merge_range(39+i*3,1,41+i*3,20,ins,muted)
    dash.merge_range('B49:U49','MANAGEMENT RECOMMENDATION  |  STUDENT: EDIT',head)
    dash.merge_range('B50:U53',recommendation,muted)
    dash.merge_range('B55:U57','Decision caution: Differences in revenue, bonus and tenure are associations. No profit, quotas, territory size or period fields exist in this dataset. Investigate those before deciding compensation.',muted)
print('Excel dashboard saved:',excel.name)


In [ ]:
# STEP 11 | Create a student Word workbook with the exact assignment checklist.
doc=Document();section=doc.sections[0];section.top_margin=Inches(.65);section.bottom_margin=Inches(.65)
doc.styles['Normal'].font.name='Aptos';doc.styles['Normal'].font.size=Pt(10)
doc.add_heading('BIAN 310  |  Assignment 4.2 Dashboard Guide',0)
doc.add_paragraph('Student: '+STUDENT_NAME+'   Assignment: '+ASSIGNMENT)
doc.add_paragraph('Dataset mode: '+('Original nine-person demonstration' if DEMO_MODE else 'Fictional 240-person practice'))
doc.add_heading('Requirements checklist',1)
for item in ['At least four explicit business questions','At least three clearly labeled KPI values',
             'At least three different useful visualizations','Three insight statements that explain the findings',
             'One evidence-based management recommendation','One dashboard title and consistent, readable formatting',
             'One-page printable Excel dashboard wherever practical','Optional Google Looker Studio recreation']:
    doc.add_paragraph(item,style='List Bullet')
doc.add_heading('Four business questions',1)
for q in business_questions:doc.add_paragraph(q)
doc.add_heading('Computed evidence (verify in Excel)',1)
tab=doc.add_table(rows=1,cols=2);tab.style='Light Shading Accent 1';tab.rows[0].cells[0].text='KPI';tab.rows[0].cells[1].text='Value'
for key,val in kpis.items():
 row=tab.add_row().cells;row[0].text=key;row[1].text=f'{val:,.0f}' if key=='Customer Accounts' else f'${val:,.2f}'
doc.add_heading('Draft statements to verify and rewrite',1)
for i,ins in enumerate(insights,1):doc.add_paragraph(ins)
doc.add_paragraph(recommendation)
doc.add_heading('Your final interpretation',1)
for caption in ['Insight 1 (sales ranking):','Insight 2 (sales vs previous bonus):',
                'Insight 3 (tenure group sales share):','One final management recommendation:']:
    doc.add_heading(caption,2);doc.add_paragraph('[Replace this prompt with your own business explanation.]')
doc.add_heading('Optional Google Looker Studio challenge',1)
for item in ['Open Looker Studio in a browser and create a blank report.',
             'Add a Google Sheets data source containing the exported Looker_Studio_Ready.csv data.',
             'Create a total-sales scorecard and one simple sales ranking chart.',
             'Confirm the units and labels, then take a screenshot or share your report link.',
             'This section is optional; the required submission remains an Excel workbook.']:
    doc.add_paragraph(item,style='List Number')
doc.add_heading('Common interpretation mistakes',1)
doc.add_paragraph('The source contains no date field, so do not claim a year-over-year trend. A tenure band with more sales might contain more employees. Correlation between prior bonuses and sales does not establish merit. Confirm values, identify missing management information, and write your own conclusion.')
word=out/'Assignment42_Student_Dashboard_Guide.docx';doc.save(word)
print('Word guide saved:',word.name)


In [ ]:
# STEP 12 | Export optional Looker-compatible CSV and AUTO-DOWNLOAD ZIP.
csv=out/'Looker_Studio_Ready.csv';clean.to_csv(csv,index=False)
pd.DataFrame([{'Question':q} for q in business_questions]).to_csv(out/'Four_Business_Questions.csv',index=False)
pd.DataFrame([{'KPI':k,'Value':v} for k,v in kpis.items()]).to_csv(out/'Five_KPIs.csv',index=False)
(out/'START_HERE.txt').write_text('BIAN310 Assignment 4.2\nStudent: '+STUDENT_NAME+'\nMode: '+('DEMO' if DEMO_MODE else 'STUDENT')+'\nOpen Excel workbook; validate 4 questions, at least 3 KPIs, 3 charts, 3 insights, and a recommendation. Rewrite DRAFT text in your own words. The optional Looker CSV is for Google Sheets import.\n',encoding='utf-8')
safe=re.sub(r'[^A-Za-z0-9_-]+','_',STUDENT_NAME).strip('_')[:35] or 'Student'
package=out/f'BIAN310_Assignment42_{safe}_{"DEMO" if DEMO_MODE else "STUDENT"}.zip'
deliver=['Assignment42_One_Page_Excel_Dashboard.xlsx','Manager_Dashboard_Preview.png',
         '01_sales_ranking.png','02_sales_bonus_scatter.png','03_sales_share_donut.png',
         'Assignment42_Student_Dashboard_Guide.docx','Looker_Studio_Ready.csv',
         'Four_Business_Questions.csv','Five_KPIs.csv','START_HERE.txt']
with ZipFile(package,'w',ZIP_DEFLATED) as z:
    for name in deliver:z.write(out/name,arcname=name)
print('COMPLETE:',package.name,'with',len(deliver),'files')
if IN_COLAB:
    from google.colab import files
    files.download(str(package))
else:print('Local ZIP:',package.resolve())
